<a href="https://colab.research.google.com/github/vineetsalar88/Tutorial/blob/master/fine_grained_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-grained 6-class classification pipeline (Colab, GPU)

**Before running:** go to `Runtime > Change runtime type > T4 GPU` so this actually uses a GPU.

Steps in this notebook:
1. Mount Google Drive (where your dataset lives)
2. Install dependencies
3. Load a self-supervised pretrained backbone (DINOv2) and fine-tune on your ~1000 trusted labels
4. Evaluate: accuracy, macro-F1, confusion matrix
5. Run the fine-tuned model over the FULL dataset to flag likely label errors and ambiguous samples
6. Save flagged CSVs and the trained model back to Drive

## 0. Check GPU is active

In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

Sun Oct 11 15:05:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Set these to wherever your data lives in Drive.
# Expected ImageFolder layout:
#   TRUSTED_DIR/class_0/*.jpg ... class_5/*.jpg   (your ~1000 trusted labels)
#   FULL_DIR/class_0/*.jpg ... class_5/*.jpg       (all data, including noisy labels)
TRUSTED_DIR = '/content/drive/MyDrive/ResearchData/GroupedData(1-946)'
FULL_DIR = '/content/drive/MyDrive/ResearchData/GroupedData(947-1128)'
OUTPUT_DIR = '/content/drive/MyDrive/dataset/outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Alternative to Drive:** if your data is a zip file, upload it directly and unzip instead:
```python
from google.colab import files
uploaded = files.upload()  # select your zip
!unzip -q your_dataset.zip -d /content/data
TRUSTED_DIR = '/content/data/trusted'
FULL_DIR = '/content/data/full'
OUTPUT_DIR = '/content/outputs'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
```
This is faster than Drive for large datasets, but it's lost when the Colab session ends — copy results to Drive at the end if you use this.

## 2. Install dependencies

In [4]:
!pip install -q scikit-learn pandas timm
# torch/torchvision already come preinstalled with GPU support on Colab

## 3. Config

In [5]:
import copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

NUM_CLASSES = 6
IMG_SIZE = 224          # bump to 384 if you have a GPU with enough memory and need finer detail
BATCH_SIZE = 32
EPOCHS_HEAD = 5          # warmup: train classifier head only
EPOCHS_FULL = 15         # then fine-tune the whole network
LR_HEAD = 1e-3
LR_FULL = 1e-5

MISMATCH_REVIEW_CSV = os.path.join(OUTPUT_DIR, 'flagged_for_review.csv')
SOFT_LABELS_CSV = os.path.join(OUTPUT_DIR, 'soft_labels.csv')
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_model.pt')

CONF_THRESHOLD = 0.70    # model must be at least this confident to flag a disagreement as a likely error
AMBIGUITY_GAP = 0.15     # top1 - top2 prob below this => treat as genuinely ambiguous, not an error

Using device: cuda


## 4. Transforms

In [6]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 5. Model: DINOv2 backbone + classification head

In [7]:
class DinoV2Classifier(nn.Module):
    """Self-supervised pretrained backbone (DINOv2) + classification head.
    Backbone is frozen initially, then unfrozen for full fine-tuning."""
    def __init__(self, num_classes, backbone_name='dinov2_vits14'):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', backbone_name)
        feat_dim = self.backbone.embed_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

    def freeze_backbone(self, freeze=True):
        for p in self.backbone.parameters():
            p.requires_grad = not freeze

## 6. Load trusted data, train/val split

In [8]:
trusted_ds = datasets.ImageFolder(TRUSTED_DIR, transform=train_tf)
class_names = trusted_ds.classes
assert len(class_names) == NUM_CLASSES, f'Expected {NUM_CLASSES} classes, found {len(class_names)}: {class_names}'

val_frac = 0.2
n_val = int(len(trusted_ds) * val_frac)
n_train = len(trusted_ds) - n_val
train_subset, val_subset = random_split(
    trusted_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
val_subset.dataset = datasets.ImageFolder(TRUSTED_DIR, transform=eval_tf)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Trusted set: {n_train} train / {n_val} val, classes={class_names}')

Trusted set: 748 train / 187 val, classes=['0', '1', '2', '3', '4', '5']


## 7. Training functions

In [9]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        n += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return total_loss / n, correct / n, all_preds, all_labels

## 8. Build model and train (two stages: head-only, then full fine-tune)

In [10]:
model = DinoV2Classifier(NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
best_val_acc = 0.0
best_state = None

print('\n[Stage 1] Training classifier head (backbone frozen)')
model.freeze_backbone(True)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
for epoch in range(EPOCHS_HEAD):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion)
    print(f'  epoch {epoch+1}/{EPOCHS_HEAD} train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

print('\n[Stage 2] Fine-tuning full network (backbone unfrozen)')
model.freeze_backbone(False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FULL)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FULL)
for epoch in range(EPOCHS_FULL):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    print(f'  epoch {epoch+1}/{EPOCHS_FULL} train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
torch.save(best_state, BEST_MODEL_PATH)
print(f'\nBest val accuracy: {best_val_acc:.4f}  (saved to {BEST_MODEL_PATH})')

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 234MB/s]



[Stage 1] Training classifier head (backbone frozen)
  epoch 1/5 train_loss=1.2178 train_acc=0.5254 val_loss=0.9519 val_acc=0.7005
  epoch 2/5 train_loss=0.9482 train_acc=0.6658 val_loss=0.8706 val_acc=0.7059
  epoch 3/5 train_loss=0.8337 train_acc=0.7112 val_loss=0.8399 val_acc=0.7059
  epoch 4/5 train_loss=0.8033 train_acc=0.7313 val_loss=0.8537 val_acc=0.7326
  epoch 5/5 train_loss=0.7222 train_acc=0.7848 val_loss=0.8438 val_acc=0.6898

[Stage 2] Fine-tuning full network (backbone unfrozen)
  epoch 1/15 train_loss=0.8746 train_acc=0.7206 val_loss=0.8318 val_acc=0.7112
  epoch 2/15 train_loss=0.6723 train_acc=0.7981 val_loss=0.7706 val_acc=0.7594
  epoch 3/15 train_loss=0.5359 train_acc=0.8864 val_loss=0.7850 val_acc=0.7540
  epoch 4/15 train_loss=0.4813 train_acc=0.9225 val_loss=0.8348 val_acc=0.7166
  epoch 5/15 train_loss=0.3983 train_acc=0.9612 val_loss=0.8014 val_acc=0.7166
  epoch 6/15 train_loss=0.3624 train_acc=0.9746 val_loss=0.7993 val_acc=0.7540
  epoch 7/15 train_loss=0.

## 9. Validation report + confusion matrix

In [11]:
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion)
print('Classification report (validation set):')
print(classification_report(val_labels, val_preds, target_names=class_names, digits=3))
print('Macro F1:', f1_score(val_labels, val_preds, average='macro'))

cm = confusion_matrix(val_labels, val_preds)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print('\nConfusion matrix (rows=true, cols=predicted):')
cm_df

Classification report (validation set):
              precision    recall  f1-score   support

           0      0.857     1.000     0.923         6
           1      0.500     0.500     0.500         6
           2      0.750     0.750     0.750        28
           3      0.854     0.745     0.796        55
           4      0.773     0.853     0.811        68
           5      0.826     0.792     0.809        24

    accuracy                          0.791       187
   macro avg      0.760     0.773     0.765       187
weighted avg      0.794     0.791     0.791       187

Macro F1: 0.7648154795696627

Confusion matrix (rows=true, cols=predicted):


,0,1,2,3,4,5
0,6,0,0,0,0,0
1,0,3,2,1,0,0
2,1,3,21,3,0,0
3,0,0,2,41,12,0
4,0,0,3,3,58,4
5,0,0,0,0,5,19


## 10. Scan the FULL dataset: flag likely label errors + ambiguous samples
Runs the fine-tuned model on every image in `FULL_DIR` (including noisy labels) and compares model prediction to the given label.

In [12]:
full_ds = datasets.ImageFolder(FULL_DIR, transform=eval_tf)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
idx_to_class = {v: k for k, v in full_ds.class_to_idx.items()}
sample_paths = [s[0] for s in full_ds.samples]

model.eval()
records, soft_label_records = [], []
ptr = 0
with torch.no_grad():
    for x, y in full_loader:
        x = x.to(DEVICE)
        probs = torch.softmax(model(x), dim=1).cpu().numpy()
        for i in range(x.size(0)):
            path = sample_paths[ptr]
            given_label = idx_to_class[y[i].item()]
            p = probs[i]
            top2_idx = np.argsort(p)[::-1][:2]
            top1, top2 = top2_idx[0], top2_idx[1]
            top1_prob, top2_prob = p[top1], p[top2]
            pred_label = idx_to_class[top1]

            is_ambiguous = (top1_prob - top2_prob) < AMBIGUITY_GAP
            disagrees = pred_label != given_label
            confident = top1_prob >= CONF_THRESHOLD

            if disagrees and confident and not is_ambiguous:
                records.append({
                    'path': path, 'given_label': given_label, 'model_prediction': pred_label,
                    'model_confidence': round(float(top1_prob), 4), 'reason': 'confident_disagreement',
                })
            elif is_ambiguous:
                soft_label_records.append({
                    'path': path, 'given_label': given_label,
                    'top1_class': idx_to_class[top1], 'top1_prob': round(float(top1_prob), 4),
                    'top2_class': idx_to_class[top2], 'top2_prob': round(float(top2_prob), 4),
                    'full_probs': json.dumps({class_names[j]: round(float(p[j]), 4) for j in range(len(class_names))}),
                })
            ptr += 1

review_df = pd.DataFrame(records).sort_values('model_confidence', ascending=False) if records else pd.DataFrame()
soft_df = pd.DataFrame(soft_label_records)

review_df.to_csv(MISMATCH_REVIEW_CSV, index=False)
soft_df.to_csv(SOFT_LABELS_CSV, index=False)

print(f'Flagged {len(review_df)} samples as likely label errors -> {MISMATCH_REVIEW_CSV}')
print(f'Found {len(soft_df)} genuinely ambiguous samples -> {SOFT_LABELS_CSV}')

if len(review_df):
    print('\nTop confusion pairs among flagged errors:')
    print(review_df.groupby(['given_label', 'model_prediction']).size().sort_values(ascending=False).head(10))

Flagged 55 samples as likely label errors -> /content/drive/MyDrive/dataset/outputs/flagged_for_review.csv
Found 16 genuinely ambiguous samples -> /content/drive/MyDrive/dataset/outputs/soft_labels.csv

Top confusion pairs among flagged errors:
given_label  model_prediction
3            4                   23
1            2                   20
5            4                    7
4            5                    3
2            4                    1
3            2                    1
dtype: int64


## 11. Preview flagged samples

In [13]:
review_df.head(20)

,path,given_label,model_prediction,model_confidence,reason
41,/content/drive/MyDrive/ResearchData/GroupedDat...,3,4,0.9863,confident_disagreement
40,/content/drive/MyDrive/ResearchData/GroupedDat...,3,4,0.9820,confident_disagreement
39,/content/drive/MyDrive/ResearchData/GroupedDat...,3,4,0.9770,confident_disagreement
49,/content/drive/MyDrive/ResearchData/GroupedDat...,5,4,0.9767,confident_disagreement
52,/content/drive/MyDrive/ResearchData/GroupedDat...,5,4,0.9767,confident_disagreement
10,/content/drive/MyDrive/ResearchData/GroupedDat...,1,2,0.9729,confident_disagreement
18,/content/drive/MyDrive/ResearchData/GroupedDat...,1,2,0.9694,confident_disagreement
37,/content/drive/MyDrive/ResearchData/GroupedDat...,3,4,0.9566,confident_disagreement
35,/content/drive/MyDrive/ResearchData/GroupedDat...,3,4,0.9543,confident_disagreement
53,/content/drive/MyDrive/ResearchData/GroupedDat...,5,4,0.9517,confident_disagreement


## 12. Done
Results are saved in `OUTPUT_DIR` on your Drive:
- `best_model.pt` — fine-tuned model weights
- `flagged_for_review.csv` — likely label errors, sorted by model confidence (review these manually, correct real errors, then retrain)
- `soft_labels.csv` — genuinely ambiguous samples with full class probabilities (use as soft targets / distillation targets instead of a forced hard label)